In [ ]:
import pandas as pd
import geopandas as gpd
from geopy.geocoders import Nominatim
from shapely.geometry import Point
from tqdm import tqdm
from sklearn.cluster import DBSCAN
import numpy as np
import matplotlib.pyplot as plt

# Load student data
df_students = pd.read_csv('/content/StudentAddresses-2016-2024.csv')

# Combine address columns to create full address for geocoding, including unit
df_students['full_address'] = (
    df_students['6a. street #'].astype(str) + ' ' +
    df_students['6b. street name'] + ' ' +
    df_students['6c. street suffix'].fillna('') + ' ' +
    df_students['6d. unit #'].fillna('') + ' ' +
    df_students['6e. zip'].astype(str)
)

# Initialize geolocator
geolocator = Nominatim(user_agent="boston_analysis")
tqdm.pandas()  # for progress bar

# Geocode each address
def geocode_address(address):
    try:
        location = geolocator.geocode(address)
        return pd.Series([location.latitude, location.longitude] if location else [None, None])
    except:
        return pd.Series([None, None])

# Apply geocoding
df_students[['latitude', 'longitude']] = df_students['full_address'].progress_apply(geocode_address)

# Load 311 data
df_311 = pd.read_csv('/content/combined_file.csv')

# Convert 311 data to GeoDataFrame
gdf_311 = gpd.GeoDataFrame(
    df_311.dropna(subset=['latitude', 'longitude']),
    geometry=gpd.points_from_xy(df_311.longitude, df_311.latitude),
    crs="EPSG:4326"
)

# Convert student data to GeoDataFrame
gdf_students = gpd.GeoDataFrame(
    df_students.dropna(subset=['latitude', 'longitude']),
    geometry=gpd.points_from_xy(df_students.longitude, df_students.latitude),
    crs="EPSG:4326"
)

# Function to link 311 data by radius around student addresses
def link_by_radius(gdf_311, gdf_students, radius=0.25):
    gdf_311 = gdf_311.to_crs(epsg=3395)  # Convert to Mercator projection for meters
    gdf_students = gdf_students.to_crs(epsg=3395)

    radius_meters = radius * 1609.34  # Convert miles to meters
    linked_data = []

    for _, student_row in gdf_students.iterrows():
        nearby_311 = gdf_311[gdf_311.geometry.within(student_row.geometry.buffer(radius_meters))]
        linked_data.append({
            'student_id': student_row['full_address'],
            'nearby_311_count': len(nearby_311),
            'nearby_311_requests': nearby_311['case_enquiry_id'].tolist()
        })

    return pd.DataFrame(linked_data)

# Link 311 data within a radius
linked_by_radius = link_by_radius(gdf_311, gdf_students)

# Linking by zip code
gdf_311['zip_code'] = df_311['location_zipcode']
gdf_students['zip_code'] = df_students['6e. zip']

# Link by zip code and count occurrences
linked_by_zip = gdf_311.groupby('zip_code').size().reset_index(name='311_request_count')
linked_by_zip.columns = ['zip_code', '311_request_count']
print("Linked by Zip Code:\n", linked_by_zip)

# Clustering with DBSCAN
combined_gdf = pd.concat([gdf_311[['geometry']], gdf_students[['geometry']]], ignore_index=True).to_crs(epsg=3395)
coords = np.array([(point.x, point.y) for point in combined_gdf.geometry])

db = DBSCAN(eps=500, min_samples=5).fit(coords)  # eps in meters
combined_gdf['cluster'] = db.labels_

# Cluster analysis
cluster_counts = combined_gdf['cluster'].value_counts().reset_index()
cluster_counts.columns = ['cluster', 'count']
print("Cluster Analysis:\n", cluster_counts)

# Plot clusters
fig, ax = plt.subplots(figsize=(12, 12))
combined_gdf[combined_gdf['cluster'] != -1].plot(column='cluster', cmap='viridis', legend=True, ax=ax)
plt.title("Clusters of 311 Requests and Student Addresses in Boston")
plt.show()
